<a href="https://colab.research.google.com/github/kwx4957/study/blob/master/cloudnet%40/03.llm-serving/week5/profiler_recipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pytorch Profile



In [9]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

In [1]:
import torch
import torchvision.models as models
from torch.profiler import profile, ProfilerActivity, record_function

In [2]:
device = "cuda"

# ImageNet 분류용 사전 정의된 ResNet-18 신경망 모델 구조 인스턴스 생성
model = models.resnet18().to(device)

# 모델 입력 규격에 맞춘 가상의 더미(Dummy) 이미지 텐서 생성
# 인자 값은 한 번에 처리할 이미지, 색상 RGB 채널, 이미지 가로 픽셀, 이지지 세로 필셀
inputs = torch.randn(5, 3, 224, 224).to(device)

In [3]:
# Warmup으로 초기 1회성 설정 작업에 따른 오버헤드를 프오파일링 측정 대상 제외
model(inputs)
torch.cuda.synchronize()

In [4]:
# 프로파일러 세션 시작, with 내 모든 pytorch 연산 및 cuda 커널 호출 인터럽트
with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA
    ],
    record_shapes=True
) as prof:
    with record_function("model_inference"):
        model(inputs)
        torch.cuda.synchronize()

In [13]:
# 프로파일러가 측정한 연산 데이터를 Operator Name별로 그룹화해 합산한 뒤
# GPU 실행 시간 기준으로 상위 10개를 요약 표 출력
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

# cpu
# print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
           torch/nn/modules/module.py(1782): _call_impl         1.23%     353.087us       145.59%      41.674ms     548.345us       0.000us         0.00%      37.783ms     497.148us            76  
                tornado/platform/asyncio.py(211): start         0.01%       2.740us       149.89%      42.906ms      14.302ms       0.000us         0.00%      10.224ms       3.408ms             3  
         

1. Self CUDA time total: 9.701ms
    - T4 GPU에서 모델 추론 커널들이 실제로 연산하는 데 걸린 총 순수 시간은 약 9.7밀리초입니다.
2. Self CPU time total: 28.625ms
    - CPU가 파이썬 코드를 실행하고 GPU 커널을 던지는 데 걸린 시간은 약 28.6밀리초입니다.


주요 항목 의미
1. torch/nn/modules/module.py(1782): _call_impl (# of Calls: 76)
    - PyTorch의 모든 신경망 모듈(Conv2d, BatchNorm2d, ReLU 등)이 호출될 때 실행되는 진입점 함수입니다.
    - ResNet-18 내부의 다양한 서브 모듈들이 총 76번 실행되었으며, 이 함수 아래에서 발생한 전체 GPU 연산 누적 시간(CUDA total)은 37.783ms 중첩 호출 포함 누적치
2. colab_kernel_launcher.py, ipykernel, tornado, asyncio
    - Google Colab 및 Jupyter 노트북이 웹 인터페이스와 통신하며 셀 코드를 백그라운드 이벤트 루프에서 실행시키는 시스템 래퍼 함수
    - 실제 딥러닝 연산 시간은 거의 먹지 않으며, 추론 코드 전체를 감싸고 있는 껍데기 프로세스일 뿐이므로 최적화 대상이 아닙니다.

>  CPU total %가 100%를 초과하는 이유 (145~149%)
>> 상위 모듈이 하위 모듈을 호출할 때 시간이 중복 합산되는 호출 스택 구조상 발생하는 일반적인 현상

In [14]:
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cuda_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
           torch/nn/modules/module.py(1782): _call_impl         1.23%     353.087us       145.59%      41.674ms     548.345us       0.000us         0.00%      37.783ms     497.148us            76  
                tornado/platform/asyncio.py(211): start         0.01%       2.740us       149.89%      42.906ms      14.302ms       0.000us         0.00%      10.224ms       3.408ms             3  
         

파이썬 오버헤드의 대부분이 cpu의 시간, gpu 연산은 단 9.7ms만에 완료되었다.

즉, 현재 추론으로는 cpu-boud의 오버헤드이다.

In [15]:
with profile(activities=activities) as prof:
    model(inputs)

prof.export_chrome_trace("trace.json")

In [17]:
# 순수 GPU 시간 기준 정렬
sort_by_keyword = "self_" + device + "_time_total"

# Python 파일 및 라인에서 어느 줄이 GPU 순수 연산 시간을 가장 많이 잡아먹었는지 역추적
with profile(
    activities=activities,
    with_stack=True,
    experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
) as prof:
    model(inputs)

# Print aggregated stats
print(prof.key_averages(group_by_stack_n=5).table(sort_by=sort_by_keyword, row_limit=2))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  -----------------------------------------------------------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  Source Location                                                    
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  -----------------------------------------------------------------  
                                aten::cudnn_convolution         0.20%      63.251us         0.30%      93.185us      93.185us     781.207us         8.06%     781.207us     781.207us             1  <built-

1. 전체 순수 GPU 시간 (Self CUDA time total): 9.693ms
    - T4 GPU에서 모델 추론 전체에 걸린 실제 시간
2. 상위 2개 레이어의 GPU 점유율: 총 16.12% (약 1.56ms)
    - 단 2개의 합성곱 레이어가 전체 추론 시간 중 약 16%를 차지
    - Conv2d_16 레이어 : 전체 8%
    - Conv2d_18 : 전체 8%

In [11]:
from torch.profiler import schedule

# 프로파일러를 수천 번의 스텝 내내 켜두면 트레이스 파일 용량이 수 기가바이트로 폭증하고 프로파일러 자체 오버헤드로 인해 학습 속도가 급격히 느려집니다.
# 이 스케줄러를 쓰면 원하는 특정 구간의 스텝만 골라서 주기적으로 가볍게 측정할 가능
my_schedule = schedule(skip_first=10, wait=5, warmup=1, active=3, repeat=2)

In [18]:
sort_by_keyword = "self_" + device + "_time_total"


def trace_handler(p):
    output = p.key_averages().table(sort_by=sort_by_keyword, row_limit=10)
    print(output)
    p.export_chrome_trace("/tmp/trace_" + str(p.step_num) + ".json")


with profile(
    activities=activities,
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=2),
    on_trace_ready=trace_handler,
) as p:
    for idx in range(8):
        model(inputs)
        p.step()

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      19.671ms        83.58%      19.671ms       9.836ms             2  
                                aten::cudnn_convolution         6.56%       1.564ms         9.50%       2.263ms      56.570us      15.801ms        67.14%      15.801ms     395.020us            40  
_5x_cudnn

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:300: UserWarning: Warning: Profiler clears events at the end of each cycle. Only events from the current cycle will be reported. To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      19.669ms        80.94%      19.669ms       9.834ms             2  
                                aten::cudnn_convolution         6.26%       1.531ms         9.18%       2.246ms      56.139us      15.799ms        65.02%      15.799ms     394.983us            40  
_5x_cudnn

schedule(wait=1, warmup=1, active=2) 스케줄러에 따라 1사이클당 4스텝(1 + 1 + 2)씩 동작.

총 8스텝 동안 2번의 사이클이 돌아 요약 표와 트레이스 파일이 각각 2회씩 출력 및 저장되었다.

## Nsight System

In [17]:
# perf_event_paranoid가 최소 레벨 2 이하어여 한다.
cat /proc/sys/kernel/perf_event_paranoid

# 1) 2보다 클 경우
# sudo sh -c 'echo 2 >/proc/sys/kernel/perf_event_paranoid'

# 2) 영구적 적용
# sudo sh -c 'echo kernel.perf_event_paranoid=2 > /etc/sysctl.d/local.conf'

2


In [22]:
# 커널 버전 조회. 최소 4.3 이상
!uname -a

# gilbc >= 2.17
!ldd --version

Linux 87b2fd787b53 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
ldd (Ubuntu GLIBC 2.35-0ubuntu3.8) 2.35
Copyright (C) 2022 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
Written by Roland McGrath and Ulrich Drepper.


In [3]:
!apt update
!apt install -y --no-install-recommends gnupg
!echo "deb http://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo "$DISTRIB_RELEASE" | tr -d .)/$(dpkg --print-architecture) /" | tee /etc/apt/sources.list.d/nvidia-devtools.list
!apt-key adv --fetch-keys http://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub
!apt update
!apt install nsight-systems-cli

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driv

In [4]:
 # nsgiht 활성화 기능 확인 및 상태 확인
 !nsys status -e

NVIDIA Nsight Systems version 2026.4.1.191-264138605071v0

General Check
- Platform: Linux
- Timestamp counter supported: Yes

CPU Profiling Environment Check
- Root privilege: enabled
- Linux Kernel Paranoid Level: 2 (some features maybe not available)
- Linux Distribution: Ubuntu
- Linux Kernel Version (6.6.122+): OK
- Linux perf_event_open syscall available: OK
- Sampling trigger event available: OK
- Intel(c) Last Branch Record support: Not Available
- CPU Profiling Environment (process-tree): OK
- CPU Profiling Environment (system-wide): OK

See the product documentation at https://docs.nvidia.com/nsight-systems for more information,
including information on how to set the Linux Kernel Paranoid Level.


In [5]:
# 1. 프로파일링할 샘플 코드 작성
script = """
import torch
import torchvision.models as models

device = 'cuda'
model = models.resnet18().to(device)
inputs = torch.randn(16, 3, 224, 224).to(device)

# Warmup
for _ in range(3):
    _ = model(inputs)

# Profiling Target
for _ in range(10):
    _ = model(inputs)
torch.cuda.synchronize()
"""
with open("test_profile.py", "w") as f:
    f.write(script)

In [6]:
# 2. nsys 실행하여 앞서 python 스크립트를 실행 후 gpu 커널 및 cpu 타임라임 데이터를 기록 및 분석한다.
!nsys profile \           # 프로파일링 모드 실행
    --trace=cuda,nvtx \   # 이벤트 종류 지정
    --stats=true \        # 통계 및 요약 터미널 출력
    -o my_report \        # 저장할 파일 명
    --force-overwrite true \ # 동일 파일 존재 시 오버라이드
    python test_profile.py

Generating '/tmp/nsys-root/nsys-report-47be.qdstrm'
[1/7] [========================100%] my_report.nsys-rep
[2/7] [========================100%] my_report.sqlite
[3/7] Executing 'nvtx_sum' stats report
SKIPPED: /content/my_report.sqlite does not contain NV Tools Extension (NVTX) data.
[4/7] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)       Med (ns)      Min (ns)     Max (ns)    StdDev (ns)            Name           
 --------  ---------------  ---------  -------------  -------------  -----------  -----------  -----------  -------------------------
     54.7      227,051,993      1,742      130,339.8        8,640.5        4,335   51,692,410  2,025,025.9  cudaLaunchKernel         
     36.1      149,932,308          1  149,932,308.0  149,932,308.0  149,932,308  149,932,308          0.0  cudaDeviceSynchronize    
      3.1       12,905,423        123      104,922.1        4,981.0        3,605    2,063,574    374,524.4  cudaMemcpyAsync          

In [7]:
# 3. ncu 실행 (보고서 파일: ncu_report.ncu-rep 생성)
# Nsight Compute로 개별 CUDA 커널의 하드웨어 메트릭(SM 점유율, 메모리 대역폭, Roofline 모델 등)을 현미경 수준으로 심층 프로파일링
# 분석에 상당히 오랜 시간이 걸린다.

# !ncu \
#     --set full \
#     -o ncu_report \
#     --force-overwrite \
#     python test_profile.py

!ncu \
    --set detailed \
    -k "regex:.*gemm.*|.*conv.*" \
    -c 1 \
    -o ncu_gemm_report \
    --force-overwrite \
    python test_profile.py

==PROF== Connected to process 13001 (/usr/bin/python3.13)
==PROF== Profiling "volta_sgemm_128x64_nn": 0%....50%....100% - 19 passes
==PROF== Disconnected from process 13001
==PROF== Report: /content/ncu_gemm_report.ncu-rep


In [9]:
# 런타임 > 세션 다시 시작 필요
!pip install -q -U vllm transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [1]:
import vllm
import torch

print("vLLM version:", vllm.__version__)
print("CUDA Available:", torch.cuda.is_available())

vLLM version: 0.28.0
CUDA Available: True


In [8]:
!pip uninstall -y torchaudio
# !pip install --no-cache-dir torchaudio --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


하단의 코드 실행 시 `RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}`

다음의 에러가 발생한다. 해당 에러는 vLLM 최신 버전이 켜진 상태에서 python 3.13으로 실행하게 될 경우 EngineCore 서브프로세스를 띄우지 못하고 즉시 크래시되면서 발생한다.

In [22]:
import os
import torch
from vllm import LLM, SamplingParams

# 프로파일러 트레이스 파일 저장 디렉토리 지정
os.environ["VLLM_TORCH_PROFILER_DIR"] = "/tmp/vllm_profile"

# LLM 초기화 (단일 프로세스 실행 강제)
llm = LLM(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",
    enforce_eager=True,
    gpu_memory_utilization=0.7,
    max_model_len=512
)

sampling_params = SamplingParams(temperature=0.7, max_tokens=32)
prompts = ["Explain what a GPU profiler does in two sentences:"] * 2

# 1. Warmup
_ = llm.generate(prompts, sampling_params)

# 2. vLLM 엔진 내부 프로파일러 시작 (워커 프로세스 추적)
llm.start_profile()

# 3. 측정 대상 추론
outputs = llm.generate(prompts, sampling_params)

# 4. 프로파일러 종료 (트레이스 파일 저장)
llm.stop_profile()

print("프로파일링 완료! 트레이스 파일 저장 경로:")
!ls -lh /tmp/vllm_profile

INFO 09-05 21:58:56 [api_utils.py:272] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
WARNING 09-05 21:58:56 [envs.py:2213] Unknown vLLM environment variable detected: VLLM_TORCH_PROFILER_DIR
INFO 09-05 21:58:57 [model.py:672] Resolved architecture: Qwen2ForCausalLM
WARNING 09-05 21:58:57 [model.py:2299] Casting torch.bfloat16 to torch.float16.
INFO 09-05 21:58:57 [model.py:1965] Using max model len 512
INFO 09-05 21:58:57 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 09-05 21:59:24 [utils.py:615] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [4]:
# !nohup python3 -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen2.5-0.5B-Instruct \
#     --dtype float16 \
#     --gpu-memory-utilization 0.7 \
#     --enforce-eager \
#     --max-model-len 512 &

nohup: appending output to 'nohup.out'
